# PIPE2D-1391-01 verification

Manual verification for the QA rebuild Phase 0/1 branch, to run against a real
Butler before merging.

**This notebook is read-only.** It opens the Butler with `writeable=False`, runs no
`pipetask run`, and writes nothing to any collection or repository. The only files it
creates are optional plot images under `OUTPUT_DIR`, and only if you ask for them in
section 5. Where a check genuinely needs a pipeline run, the notebook prints the command
for you to run yourself rather than running it.

## What each section establishes

| Section | Question | Needs |
|---|---|---|
| 1 | Does everything import, and does the pipeline build? | stack only |
| 2 | Do the stack-dependent tests pass? | stack only |
| 3 | **Did the registry migration move any verdict?** | a collection with `iqQaMetrics` |
| 4 | Is the new per-species dataset present and readable? | a collection from this branch |
| 5 | Do the extracted plotting functions still draw? | a collection with `dmQaResidualData` |
| 6 | Do the golden `known_good` visits pass? | a collection covering Run25 |
| 7 | Does the threshold CLI run against a real Butler? | a collection with `iqQaMetrics` |

Section 3 is the one that matters most: the ticket promises **no change to any QA
verdict**, and section 3 tests exactly that, on real numbers, without running anything.

## Configuration

Set these, then run the notebook top to bottom.

In [ ]:
# --- Butler ---------------------------------------------------------------
BUTLER_REPO = "/work/datastore"

# Any collection holding iqQaMetrics. It does NOT need to come from this branch:
# section 3a only reads the measured values, which this branch does not touch, so an
# old collection reduced months ago on main works perfectly. Section 3b additionally
# reads the stored qaStatus and says so when that makes it circular.
COLLECTION = "qaActor/reductions"

# Optional: a second collection to diff against, if you did run the pipeline twice.
# Leave as None to skip the direct two-collection comparison in section 3b.
BASELINE_COLLECTION = None

# Optional: restrict every query to these visits. None means "whatever is there".
VISITS = None  # e.g. [133025, 133028, 133031, 133034, 133037, 133040]

# Cap on how many iqQaMetrics datasets to load. None loads all of them. The gating
# comparison is per-row and cheap, so raise or remove this freely; it exists so the
# notebook stays usable as qaActor/reductions grows.
MAX_DATASETS = None

# --- Local ----------------------------------------------------------------
OUTPUT_DIR = "pipe2d1391-verification"  # only written to if you opt in, in section 5
REPO_ROOT = ".."                        # path to the drp_qa checkout from this notebook

In [ ]:
import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path(REPO_ROOT).resolve()
sys.path.insert(0, str(REPO_ROOT / "python"))

RESULTS = {}  # section -> (ok, message); summarised at the end


def record(section, ok, message):
    """Record a result and print it."""
    RESULTS[section] = (ok, message)
    mark = "PASS" if ok else ("SKIP" if ok is None else "FAIL")
    print(f"[{mark}] {section}: {message}")


def skip(section, message):
    record(section, None, message)


print(f"drp_qa checkout: {REPO_ROOT}")

## 1. Imports and pipeline build

Exercises the plotting extraction and the `FitStats` move. A broken re-export shim fails
here rather than part-way through a reduction.

In [ ]:
try:
    import pfs.drp.qa.dmCombinedResiduals  # noqa: F401
    import pfs.drp.qa.dmResiduals  # noqa: F401
    import pfs.drp.qa.imageQualityQa  # noqa: F401
    import pfs.drp.qa.plotting  # noqa: F401
    from pfs.drp.qa.iqQaPlots import plotIqTimeSeries  # legacy shim  # noqa: F401
    from pfs.drp.qa.utils.plotting import detector_palette  # legacy shim  # noqa: F401
except Exception as exc:  # noqa: BLE001
    record("1 imports", False, f"{type(exc).__name__}: {exc}")
else:
    record("1 imports", True, "all modules and legacy shims import")

In [ ]:
# --show tasks does not consume the butler, so passing -b is rejected -- and
# without -b the graph cannot resolve the PFS dimensions arm and spectrograph,
# which live in the repository's dimension config rather than the default
# universe. --show pipeline-graph is one of the forms that does use the butler.
build = subprocess.run(
    [
        "pipetask",
        "build",
        "-b",
        BUTLER_REPO,
        "-p",
        str(REPO_ROOT / "pipelines" / "drpQA.yaml"),
        "--show",
        "pipeline-graph",
    ],
    capture_output=True,
    text=True,
)
print(build.stdout or build.stderr)

expected = {"dmResiduals", "dmCombinedResiduals", "extractionQa", "extractionQaCombined", "imageQualityQa"}
missing = {label for label in expected if label not in build.stdout}
# The new output connection should appear in the resolved graph.
speciesDeclared = "iqQaSpeciesMetrics" in build.stdout

if build.returncode != 0:
    record("1 pipeline build", False, f"pipetask build exited {build.returncode}; see the output above")
else:
    notes = []
    if missing:
        # The graph resolved, which is what is being tested. A label absent from
        # the rendering is more likely a formatting difference than a missing task.
        notes.append(f"labels not found in output (check by eye): {sorted(missing)}")
    if not speciesDeclared:
        notes.append("iqQaSpeciesMetrics not seen in the graph -- confirm the new output connection")
    record(
        "1 pipeline build",
        True,
        "; ".join(notes) if notes else "graph resolves, all five labels and iqQaSpeciesMetrics present",
    )


## 2. Stack-dependent tests

`tests/test_dmResiduals.py` has never run against the real stack — only against a stub
with a hand-written `robustRms`. This is where a difference would show.

In [ ]:
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(REPO_ROOT / "tests")],
    capture_output=True,
    text=True,
    cwd=str(REPO_ROOT),
)
print(tests.stdout[-4000:] or tests.stderr[-4000:])
record("2 tests", tests.returncode == 0, "pytest passed" if tests.returncode == 0 else "pytest failed, see above")

### 3a. Old ladder vs new registry, on stored numbers

**This section does not read `qaStatus`, and needs no pipeline run.** It pulls the measured
values out of `iqQaMetrics` -- `medFwhm`, `pctFlagged`, `medDxCenter`, `traceOnly`,
`seqName` -- and puts each row through two pure functions: the gating ladder transcribed
verbatim from `main`, and the new registry. Then it compares those two outputs *to each
other*.

So the collection is only a source of realistic numbers. This branch changes no
measurement code -- the diff to `imageQualityQa.py` touches imports, connections,
docstrings, the gating block and the new species output, and nothing that computes a
metric -- so those numbers are the same whichever branch produced them. **Any collection
with `iqQaMetrics` works, including one reduced months ago.**

Any disagreement is a real regression in the migration.


In [ ]:
from lsst.daf.butler import Butler

butler = Butler(BUTLER_REPO, collections=[COLLECTION], writeable=False)  # read-only

where = ""
bind = {}
if VISITS:
    where = "visit IN (visits)"
    bind = {"visits": list(VISITS)}

refs = sorted(
    set(butler.registry.queryDatasets("iqQaMetrics", collections=[COLLECTION], where=where, bind=bind, findFirst=True)),
    key=lambda ref: (ref.dataId.get("visit", 0), str(ref.dataId.get("arm", "")), ref.dataId.get("spectrograph", 0)),
)
print(f"{len(refs)} iqQaMetrics datasets in {COLLECTION}")
if MAX_DATASETS is not None and len(refs) > MAX_DATASETS:
    print(f"  loading the first {MAX_DATASETS}; raise MAX_DATASETS to use them all")
    refs = refs[:MAX_DATASETS]

metrics = pd.concat([butler.get(ref) for ref in refs], ignore_index=True) if refs else pd.DataFrame()
if metrics.empty:
    skip("3a", "no iqQaMetrics in this collection; nothing to compare")
else:
    display(metrics[["visit", "arm", "spectrograph", "seqName", "medFwhm", "pctFlagged", "medDxCenter", "qaStatus"]].head(12))

In [ ]:
# The gating ladder exactly as it stood on `main`, transcribed for comparison.
# Do not "tidy" this: its value is being a faithful copy of what it replaced.
def legacyStatus(row, config):
    """Return the qaStatus that main's if/elif ladder would have produced."""
    medFwhm = row["medFwhm"]
    traceOnly = bool(row.get("traceOnly", False))
    pctFlagged = row["pctFlagged"]
    medDxCenter = row["medDxCenter"]
    seqName = row.get("seqName") or ""
    arm = row.get("arm") or ""

    fwhmStatus = "PASS"
    if not traceOnly and not np.isnan(medFwhm):
        if medFwhm >= config.fwhmFailThreshold:
            fwhmStatus = "FAIL"
        elif medFwhm >= config.fwhmWarnThreshold:
            fwhmStatus = "WARN"

    flagStatus = "PASS"
    if np.isfinite(pctFlagged):
        species = seqName.split(":", 1)[-1].strip() if ":" in seqName else ""
        compoundKey = f"{arm}:{species}" if species else ""
        warnThresh = config.flagRateWarnThreshold.get(compoundKey, config.flagRateWarnThreshold.get(arm, 15.0))
        failThresh = config.flagRateFailThreshold.get(compoundKey, config.flagRateFailThreshold.get(arm, 20.0))
        if pctFlagged >= failThresh:
            flagStatus = "FAIL"
        elif pctFlagged >= warnThresh:
            flagStatus = "WARN"

    dxStatus = "PASS"
    if np.isfinite(medDxCenter):
        absDx = abs(medDxCenter)
        if absDx >= config.dxCenterFailThreshold:
            dxStatus = "FAIL"
        elif absDx >= config.dxCenterWarnThreshold:
            dxStatus = "WARN"

    level = {"PASS": 0, "WARN": 1, "FAIL": 2}
    return max((fwhmStatus, flagStatus, dxStatus), key=lambda s: level[s])

In [ ]:
from pfs.drp.qa.imageQualityQa import ImageQualityQaConfig
from pfs.drp.qa.metrics.definitions import buildImageQualityRegistry
from pfs.drp.qa.metrics.registry import UNKNOWN, worstStatus

config = ImageQualityQaConfig()  # shipped defaults, as both branches use them
registry = buildImageQualityRegistry(config)


def registryStatus(row):
    """Return the qaStatus the new registry produces, mirroring the task."""
    arm = row.get("arm") or ""
    seqName = row.get("seqName") or ""
    species = seqName.split(":", 1)[-1].strip() if ":" in seqName else ""
    flagRateKeys = (f"{arm}:{species}" if species else "", arm)
    traceOnly = bool(row.get("traceOnly", False))
    # Trace quanta gate medFwhm on their own per-arm keys, and their pctFlagged
    # is suppressed, so the arc thresholds never reach a fiber-profile width.
    traceKeys = (f"trace:{arm}", "trace") if traceOnly else ()
    return worstStatus(
        [
            registry.gate("medFwhm", row["medFwhm"], keys=traceKeys),
            registry.gate("pctFlagged", row["pctFlagged"], keys=flagRateKeys),
            registry.gate("medDxCenter", row["medDxCenter"]),
        ],
        default=UNKNOWN,
    )


In [ ]:
if metrics.empty:
    skip("3a", "no iqQaMetrics to re-gate")
else:
    compare = metrics.copy()
    compare["legacy"] = compare.apply(lambda row: legacyStatus(row, config), axis=1)
    compare["registry"] = compare.apply(registryStatus, axis=1)
    disagree = compare[compare["legacy"] != compare["registry"]]

    # One divergence is intended: this branch reports UNKNOWN where the old
    # ladder reported PASS on a quantum with no finite measurement. Separate
    # that from anything else, which would be a genuine regression.
    measured = [c for c in ("medFwhm", "pctFlagged", "medDxCenter") if c in compare.columns]
    nothingMeasured = compare[measured].isna().all(axis=1)
    intended = disagree[
        (disagree["legacy"] == "PASS")
        & (disagree["registry"] == "UNKNOWN")
        & nothingMeasured.reindex(disagree.index, fill_value=False)
    ]
    unexplained = disagree.drop(index=intended.index)

    print(f"{len(compare)} quanta | {len(disagree)} disagree "
          f"({len(intended)} intended PASS->UNKNOWN, {len(unexplained)} unexplained)")

    if unexplained.empty:
        record(
            "3a",
            True,
            f"no unintended verdict change across {len(compare)} quanta"
            + (f"; {len(intended)} moved PASS->UNKNOWN by design" if len(intended) else ""),
        )
    else:
        record("3a", False, f"{len(unexplained)} quanta changed verdict unexpectedly -- see below")
        display(
            unexplained[
                [c for c in ["visit", "arm", "spectrograph", "seqName", "medFwhm", "pctFlagged",
                             "medDxCenter", "legacy", "registry"] if c in unexplained.columns]
            ]
        )
    if not intended.empty:
        print("\nintended PASS -> UNKNOWN (nothing was measurable):")
        display(intended[[c for c in ["visit", "arm", "spectrograph", "seqName", *measured]
                          if c in intended.columns]].head(20))


In [ ]:
# How strong is that agreement? Two functions that both return PASS everywhere
# agree trivially. This reports what the comparison actually exercised: which
# threshold keys were hit, and whether any row landed on a non-PASS verdict.
if metrics.empty:
    skip("3a coverage", "nothing to summarise")
else:
    def gateKey(row):
        arm = row.get("arm") or ""
        seqName = row.get("seqName") or ""
        species = seqName.split(":", 1)[-1].strip() if ":" in seqName else ""
        return f"{arm}:{species}" if species else (arm or "(no arm)")

    compare["gateKey"] = compare.apply(gateKey, axis=1)
    breakdown = (
        compare.groupby(["gateKey", "registry"], observed=True)
        .size()
        .unstack(fill_value=0)
        .sort_index()
    )
    display(breakdown)

    verdicts = set(compare["registry"].unique())
    nonPass = int((compare["registry"] != "PASS").sum())
    traceOnly = int(compare.get("traceOnly", pd.Series(dtype=bool)).fillna(False).astype(bool).sum())
    print(f"threshold keys exercised: {compare['gateKey'].nunique()}")
    print(f"verdicts seen: {sorted(verdicts)}")
    print(f"non-PASS rows: {nonPass} of {len(compare)}   |   traceOnly rows: {traceOnly}")

    if nonPass == 0:
        record(
            "3a coverage",
            None,
            "every row is PASS, so the two functions have only been shown to agree in the "
            "easy region -- point this at a collection containing some WARN/FAIL quanta to "
            "exercise the threshold boundaries",
        )
    else:
        record(
            "3a coverage",
            True,
            f"{compare['gateKey'].nunique()} threshold keys and {nonPass} non-PASS rows exercised",
        )


### 3b. Re-gated verdicts vs what is stored

**This can only establish parity against a collection produced by one known code
version.** A production collection such as `qaActor/reductions` is reduced incrementally,
each visit by whatever was deployed at the time, so its stored `qaStatus` is a mix of
versions and possibly of configs. Disagreement there says the stored verdicts came from
something other than today's code -- which is expected, and says nothing about this
branch.

The cell below counts the RUN collections the datasets span. If there is more than one,
3b reports SKIP rather than FAIL, because a mixed baseline cannot be compared against.

**The trustworthy version of this check is 3c**: reduce the same visits twice, once on
`main` and once on this branch, into two fresh collections, and diff them. That is the
only form where both sides are known to come from one code version each:

```bash
# on main
pipetask run -p pipelines/drpQA.yaml#imageQualityQa -b $BUTLER -i $INPUT \
    -o u/$USER/qa-main -d "visit IN (133025, 133028, 133031, 133034, 133037, 133040)"
# on tickets/PIPE2D-1391-01
pipetask run -p pipelines/drpQA.yaml#imageQualityQa -b $BUTLER -i $INPUT \
    -o u/$USER/qa-branch --register-dataset-types -d "visit IN (...)"
```

Then set `BASELINE_COLLECTION = "u/$USER/qa-main"` and `COLLECTION = "u/$USER/qa-branch"`.


In [ ]:
# How many RUN collections do these datasets span? A production collection
# accumulates one run per reduction, so more than a handful means the stored
# verdicts come from several code versions and cannot serve as a baseline.
runs = pd.Series([ref.run for ref in refs]).value_counts() if refs else pd.Series(dtype=int)
print(f"{len(runs)} RUN collection(s) behind these datasets")
display(runs.head(20))

if metrics.empty:
    skip("3b", "no iqQaMetrics to compare")
else:
    mismatch = compare[compare["registry"] != compare["qaStatus"]]
    heterogeneous = len(runs) > 1

    if mismatch.empty:
        record("3b", True, f"stored qaStatus matches the registry for all {len(compare)} quanta")
    elif heterogeneous:
        # Not a failure: the baseline is a mixture, so there is nothing to be
        # parity WITH. Calling this FAIL would be reporting a defect that the
        # data cannot evidence.
        skip(
            "3b",
            f"{len(mismatch)} of {len(compare)} differ, but these datasets span {len(runs)} runs, "
            "so the stored verdicts come from several code versions -- no baseline to compare "
            "against. Use 3c with two fresh collections instead",
        )
    else:
        record(
            "3b",
            False,
            f"{len(mismatch)} quanta differ from the stored qaStatus, from a single run "
            "-- worth investigating with the diagnostics below",
        )
    if not mismatch.empty:
        display(mismatch[["visit", "arm", "spectrograph", "seqName", "qaStatus", "registry"]].head(20))


In [ ]:
# --- 3b diagnosis --------------------------------------------------------
# 3a showed the old ladder and the new registry agree. If the stored qaStatus
# disagrees with both, it was produced by neither -- so the question is what it
# WAS produced with, not whether the migration moved a boundary.
#
# First candidate, and the cheapest to rule out: the config. The cells above
# rebuild the registry from ImageQualityQaConfig() defaults, but a production
# run may carry -c overrides. Every pipetask run stores the config it used.
storedConfig = None
try:
    configRefs = list(butler.registry.queryDatasets("imageQualityQa_config", collections=[COLLECTION]))
    if configRefs:
        storedConfig = butler.get(configRefs[0])
        print(f"loaded imageQualityQa_config from {configRefs[0].run}")
except Exception as exc:  # noqa: BLE001
    print(f"could not read imageQualityQa_config: {type(exc).__name__}: {exc}")

if storedConfig is None:
    print("no stored config found; the comparison above used shipped defaults")
else:
    fields = [
        "fwhmWarnThreshold",
        "fwhmFailThreshold",
        "dxCenterWarnThreshold",
        "dxCenterFailThreshold",
        "flagRateWarnThreshold",
        "flagRateFailThreshold",
    ]
    drift = {}
    for field in fields:
        mine = getattr(config, field, None)
        theirs = getattr(storedConfig, field, None)
        mine = dict(mine) if hasattr(mine, "keys") else mine
        theirs = dict(theirs) if hasattr(theirs, "keys") else theirs
        if mine != theirs:
            drift[field] = {"defaults": mine, "as run": theirs}
    if drift:
        print("\nTHRESHOLDS DIFFER FROM DEFAULTS -- this alone explains a qaStatus mismatch:")
        for field, values in drift.items():
            print(f"  {field}:\n    defaults: {values['defaults']}\n    as run:   {values['as run']}")
    else:
        print("\nstored config matches the shipped defaults; config drift is NOT the explanation")


In [ ]:
# Re-gate with the config the collection was actually produced with, and see
# whether the 217 disagreements survive.
if storedConfig is None:
    skip("3b regated", "no stored config to re-gate with")
else:
    storedRegistry = buildImageQualityRegistry(storedConfig)

    def registryStatusAsRun(row):
        arm = row.get("arm") or ""
        seqName = row.get("seqName") or ""
        species = seqName.split(":", 1)[-1].strip() if ":" in seqName else ""
        keys = (f"{arm}:{species}" if species else "", arm)
        traceOnly = bool(row.get("traceOnly", False))
        return worstStatus(
            [
                None if traceOnly else storedRegistry.gate("medFwhm", row["medFwhm"]),
                storedRegistry.gate("pctFlagged", row["pctFlagged"], keys=keys),
                storedRegistry.gate("medDxCenter", row["medDxCenter"]),
            ]
        )

    compare["registryAsRun"] = compare.apply(registryStatusAsRun, axis=1)
    compare["legacyAsRun"] = compare.apply(lambda row: legacyStatus(row, storedConfig), axis=1)

    stillDiffers = compare[compare["registryAsRun"] != compare["qaStatus"]]
    ladderAlsoDiffers = compare[compare["legacyAsRun"] != compare["qaStatus"]]

    print(f"with the config as run: {len(stillDiffers)} of {len(compare)} still differ from stored")
    print(f"the OLD ladder with the same config differs on {len(ladderAlsoDiffers)}")

    if len(stillDiffers) == 0:
        record("3b regated", True, "config drift fully explains it; the migration changes nothing")
    elif len(stillDiffers) == len(ladderAlsoDiffers):
        record(
            "3b regated",
            True,
            f"{len(stillDiffers)} rows still differ, but the OLD ladder disagrees with the stored "
            "verdicts on exactly the same rows -- so this is not the migration, it is something "
            "about how those verdicts were produced (see the breakdown below)",
        )
    else:
        record(
            "3b regated",
            False,
            f"{len(stillDiffers)} differ under the registry vs {len(ladderAlsoDiffers)} under the "
            "old ladder -- the migration IS implicated; inspect the rows below",
        )


In [ ]:
# Characterise whatever still disagrees: direction, and which quanta.
suspect = compare[compare.get("registryAsRun", compare["registry"]) != compare["qaStatus"]]
if suspect.empty:
    print("nothing left to characterise")
else:
    recomputed = "registryAsRun" if "registryAsRun" in compare.columns else "registry"
    print("stored qaStatus (rows) vs recomputed (columns):")
    display(pd.crosstab(suspect["qaStatus"], suspect[recomputed]))

    for field in ("arm", "obsType", "seqName", "traceOnly"):
        if field in suspect.columns:
            print(f"\nby {field}:")
            display(suspect[field].value_counts())

    print("\nsample of the disagreeing rows:")
    cols = [c for c in ["visit", "arm", "spectrograph", "seqName", "obsType", "traceOnly",
                        "medFwhm", "pctFlagged", "medDxCenter", "nLines", "qaStatus", recomputed]
            if c in suspect.columns]
    display(suspect[cols].head(25))


In [ ]:
# 3c. If you did run the pipeline twice, diff the two collections directly.
if BASELINE_COLLECTION is None:
    skip("3c", "BASELINE_COLLECTION not set; skipping the two-collection diff")
else:
    baseButler = Butler(BUTLER_REPO, collections=[BASELINE_COLLECTION], writeable=False)
    baseRefs = set(baseButler.registry.queryDatasets("iqQaMetrics", collections=[COLLECTION], where=where, bind=bind, findFirst=True))
    baseline = pd.concat([baseButler.get(ref) for ref in baseRefs], ignore_index=True)

    key = ["visit", "arm", "spectrograph"]
    cols = ["qaStatus", "medFwhm", "medDxCenter", "pctFlagged"]
    left = baseline.set_index(key)[cols].sort_index()
    right = metrics.set_index(key)[cols].sort_index()
    common = left.index.intersection(right.index)
    diff = left.loc[common].compare(right.loc[common])

    if diff.empty:
        record("3c", True, f"{len(common)} quanta identical across the two collections")
    else:
        record("3c", False, f"{len(diff)} quanta differ -- see below")
        display(diff)

## 4. The new per-species dataset

`iqQaSpeciesMetrics` replaces the ragged `fitSpeciesXRms_<species>` columns. Two things to
confirm: the long frames concatenate to a stable schema, and `imageQualityLogQa.py` finds
them again — it read the removed columns, which review caught.

In [ ]:
from pfs.drp.qa.metrics.longFormat import LONG_COLUMNS

speciesRefs = set()
try:
    speciesRefs = set(butler.registry.queryDatasets("iqQaSpeciesMetrics", collections=[COLLECTION], where=where, bind=bind, findFirst=True))
except Exception as exc:  # noqa: BLE001
    print(f"query failed: {type(exc).__name__}: {exc}")

if not speciesRefs:
    skip("4 species dataset", "no iqQaSpeciesMetrics here; expected for a collection from main")
else:
    frames = [butler.get(ref) for ref in speciesRefs]
    combined = pd.concat(frames, ignore_index=True)
    schemaOk = tuple(combined.columns) == LONG_COLUMNS
    padded = combined["value"].isna().sum()
    print(f"{len(frames)} datasets, {len(combined)} rows, species seen: {sorted(combined['description'].unique())}")
    display(combined.head(12))

    if not schemaOk:
        record("4 species dataset", False, f"unexpected columns: {tuple(combined.columns)}")
    elif padded:
        record("4 species dataset", False, f"{padded} NaN values -- long format should not pad")
    else:
        record("4 species dataset", True, f"stable schema across {len(frames)} quanta, no NaN padding")

In [ ]:
# The report path that review found broken. Requires a visit present in the collection.
reportVisit = int(metrics["visit"].iloc[0]) if not metrics.empty else None
reportSpec = int(metrics["spectrograph"].iloc[0]) if not metrics.empty else None

if reportVisit is None:
    skip("4 report", "no visit available")
else:
    cmd = [
        sys.executable,
        str(REPO_ROOT / "bin.src" / "imageQualityLogQa.py"),
        "--butler", BUTLER_REPO,
        "--collection", COLLECTION,
        "--visit", str(reportVisit),
        "--spectrograph", str(reportSpec),
    ]
    print(" ".join(cmd), "\n")
    report = subprocess.run(cmd, capture_output=True, text=True)
    print(report.stdout[-4000:] or report.stderr[-4000:])
    if report.returncode != 0:
        record("4 report", False, f"imageQualityLogQa.py exited {report.returncode}")
    else:
        record(
            "4 report",
            True,
            "ran; CHECK BY EYE that per-species residuals appear -- an empty species "
            "section means the iqQaSpeciesMetrics merge is not firing",
        )

## 5. Plotting

The plotting functions moved to `pfs.drp.qa.plotting` and now take DataFrames and a
`DetectorGeometry` rather than a `DetectorMap`. Regenerating a figure from stored data
exercises that, without running the pipeline.

Set `SAVE_FIGURES = True` if you want them written to `OUTPUT_DIR`; otherwise nothing is
written to disk.

In [ ]:
SAVE_FIGURES = False

from pfs.drp.qa.plotting import DetectorGeometry, plot_detectormap_residuals

dataRefs = sorted(
    set(butler.registry.queryDatasets("dmQaResidualData", collections=[COLLECTION], where=where, bind=bind, findFirst=True)),
    key=lambda ref: (ref.dataId.get("visit", 0), str(ref.dataId.get("arm", ""))),
)
statRefs = {
    (ref.dataId.get("visit"), ref.dataId.get("arm"), ref.dataId.get("spectrograph")): ref
    for ref in butler.registry.queryDatasets("dmQaResidualStats", collections=[COLLECTION], where=where, bind=bind, findFirst=True)
}

if not dataRefs:
    skip("5 plotting", "no dmQaResidualData in this collection")
else:
    ref = dataRefs[0]
    dataId = ref.dataId
    statRef = statRefs.get((dataId.get("visit"), dataId.get("arm"), dataId.get("spectrograph")))
    arcData = butler.get(ref)
    visitStats = butler.get(statRef) if statRef is not None else None

    if visitStats is None:
        skip("5 plotting", "matching dmQaResidualStats not found")
    else:
        detectorMap = butler.get("detectorMap", dataId=dataId)
        geometry = DetectorGeometry.fromDetectorMap(detectorMap)
        print(f"{dict(dataId.mapping)} -> {geometry}")

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            figure = plot_detectormap_residuals(arcData, visitStats, geometry)

        if figure is None:
            record("5 plotting", False, "plot_detectormap_residuals returned None")
        else:
            record("5 plotting", True, "figure drawn from stored data via DetectorGeometry")
            if SAVE_FIGURES:
                Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
                out = Path(OUTPUT_DIR) / "dmResiduals.png"
                figure.savefig(out, dpi=120)
                print(f"wrote {out}")
            display(figure)

## 6. The golden visit set

If this collection covers the Run25 stable sequence, every `known_good` detector should
report PASS. **A failure here is the most interesting result in the notebook**: either a
threshold is wrong, or the "assumed stable" premise does not hold — and either way you
want to know before thresholds are re-derived from those visits.

The comparison uses the **recomputed** verdict, not the stored `qaStatus`, for the reason
3b gives: in an incrementally-reduced collection the stored verdict came from whatever code
was deployed at the time, so checking it against the golden set would be a statement about
the collection's history rather than about today's thresholds. The stored value is shown
alongside, and where the two differ that is itself worth a look.


In [ ]:
from pfs.drp.qa.metrics.goldenVisits import loadGoldenVisits

golden = loadGoldenVisits(REPO_ROOT / "tests" / "data" / "goldenVisits.yaml")
print(f"{len(golden.knownGood)} known_good entries, {len(golden.knownBad)} known_bad "
      f"({len(golden.confirmedBad)} confirmed)")

# Prefer the verdict recomputed with the config the collection was produced
# with; fall back to shipped defaults, then to the stored value.
verdictColumn = next(
    (c for c in ("registryAsRun", "registry") if "compare" in dir() and c in compare.columns),
    None,
)

if metrics.empty or verdictColumn is None:
    skip("6 golden set", "no recomputed verdicts available; run section 3 first")
else:
    rows = []
    for row in compare.to_dict("records"):
        expected = golden.expectationFor(
            int(row["visit"]),
            arm=row.get("arm"),
            spectrograph=int(row["spectrograph"]) if pd.notna(row.get("spectrograph")) else None,
            seqType=row.get("seqName"),
        )
        if expected is None:
            continue  # not in the golden set: no expectation, which is not a PASS
        rows.append({**row, "expected": expected, "recomputed": row[verdictColumn]})

    checked = pd.DataFrame(rows)
    if checked.empty:
        skip("6 golden set", "none of these visits are in the golden set")
    else:
        mismatched = checked[checked["expected"] != checked["recomputed"]]
        unassessed = mismatched[mismatched["recomputed"] == "UNKNOWN"]
        wrong = mismatched.drop(index=unassessed.index)
        print(f"{len(checked)} quanta carry a golden-set expectation "
              f"(verdict recomputed via '{verdictColumn}')")

        drifted = checked[checked["recomputed"] != checked["qaStatus"]]
        if not drifted.empty:
            print(f"note: {len(drifted)} of them have a stored qaStatus that differs from the "
                  "recomputed one -- expected in a collection reduced over time")

        if not unassessed.empty:
            print(f"{len(unassessed)} golden-set quanta report UNKNOWN -- not a wrong verdict, "
                  "but nothing could be measured there, so they establish nothing")
            display(unassessed[[c for c in ["visit", "arm", "spectrograph", "seqName", "expected"]
                                if c in unassessed.columns]])

        if wrong.empty and unassessed.empty:
            record("6 golden set", True, f"all {len(checked)} quanta match their expectation")
        elif wrong.empty:
            record("6 golden set", None,
                   f"no wrong verdicts, but {len(unassessed)} quanta could not be measured")
        else:
            record("6 golden set", False, f"{len(wrong)} quanta do not match -- see below")
            display(
                wrong[
                    [c for c in ["visit", "arm", "spectrograph", "seqName", "expected", "recomputed",
                                 "qaStatus", "medFwhm", "pctFlagged", "medDxCenter"] if c in wrong.columns]
                ]
            )


In [ ]:
# A PASS earned by measuring nothing is not a pass. When every metric is NaN
# each gate returns "not judged" and worstStatus falls through to its default,
# so the quantum reports PASS -- and a known_good expectation is satisfied
# vacuously. This is pre-existing behaviour, not something this branch
# introduces, but the golden set cannot tell the two apart, so surface it.
if metrics.empty or verdictColumn is None:
    skip("6 vacuous passes", "no recomputed verdicts available")
else:
    measured = [c for c in ("medFwhm", "pctFlagged", "medDxCenter") if c in compare.columns]
    nothingMeasured = compare[measured].isna().all(axis=1)
    vacuous = compare[nothingMeasured & (compare[verdictColumn] == "PASS")]

    print(f"{int(nothingMeasured.sum())} of {len(compare)} quanta have no finite measurement at all")
    if vacuous.empty:
        record("6 vacuous passes", True, "every PASS rests on at least one real measurement")
    else:
        inGolden = [
            visit for visit in vacuous["visit"].unique()
            if golden.expectationFor(int(visit)) is not None
        ]
        record(
            "6 vacuous passes",
            None,
            f"{len(vacuous)} quanta report PASS with no finite measurement"
            + (f", covering golden-set visits {sorted(inGolden)}" if inGolden else "")
            + " -- pre-existing: nothing was measured, so nothing was judged, and the "
              "default is PASS. Not introduced by this branch",
        )
        cols = [c for c in ["visit", "arm", "spectrograph", "seqName", "obsType", "traceOnly",
                            "nLines", *measured, verdictColumn] if c in vacuous.columns]
        display(vacuous[cols])


## 7. Threshold calibration CLI

Reads the collection and prints suggestions. It writes nothing. A non-zero exit is
meaningful, not a crash: it refuses to hand you thresholds it cannot stand behind — too
few samples, or known-bad data that does not cross the suggested FAIL.

In [ ]:
cmd = [
    sys.executable,
    str(REPO_ROOT / "bin.src" / "calibrateQaThresholds.py"),
    "-b", BUTLER_REPO,
    "-c", COLLECTION,
    "--metric", "medFwhm",
    "--metric", "pctFlagged",
    "--group-by", "arm",
]
print(" ".join(cmd), "\n")
calib = subprocess.run(cmd, capture_output=True, text=True)
print(calib.stdout)
print(calib.stderr, file=sys.stderr)

record(
    "7 threshold CLI",
    calib.returncode in (0, 1),
    f"exited {calib.returncode} "
    + ("(suggestions produced)" if calib.returncode == 0 else "(refused to stand behind the numbers -- read the reason above)"),
)

## Summary

In [ ]:
summary = pd.DataFrame(
    [
        {"section": name, "result": "PASS" if ok else ("SKIP" if ok is None else "FAIL"), "detail": message}
        for name, (ok, message) in RESULTS.items()
    ]
)
display(summary)

failed = [name for name, (ok, _) in RESULTS.items() if ok is False]
skipped = [name for name, (ok, _) in RESULTS.items() if ok is None]

if failed:
    print(f"FAILED: {', '.join(failed)}")
    print("Blocking: 1 (pipeline builds), 2 (tests), 3 (no verdict moved).")
else:
    print("Nothing failed.")

if skipped:
    print(f"\nSKIPPED, which is not a pass: {', '.join(skipped)}")
    print("A skipped section established nothing. Read its reason before treating it as clear.")
